# SmolLM3 -> GGUF: F16 + imatrix + Q4_K_M + Q8_0 + eval + gated HF upload
Self-sufficient Save & Run notebook. Zero interactive prompts; any failure stops loudly instead of hanging.
Manual inputs (3): 1) Secrets: add HF_TOKEN (write token) and ATTACH it. 2) Verify BASE_REPO in cell 2. 3) Set REPO_NAME in cell 2.
Runtime: Internet ON, GPU T4 x2 (work is CPU-only; this matches the proven run's RAM/disk). Total ~8h (4B size), imatrix is the long pole.

In [ ]:
import os, sys, time, json, hashlib, random, re, shutil, subprocess
W = "/kaggle/working"
os.chdir(W)
os.environ["PIP_NO_INPUT"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
# ---------------- SET THESE ----------------
BASE_REPO    = "Qwen/Qwen3-4B"
REPO_NAME    = "Qwen3-4B-Q4_K_M-GGUF"
REPO_PRIVATE = False
SEED, CALIB_LINES, BAR_PCT = 42, 3000, 5.0
QUANTS = ["Q4_K_M", "Q8_0"]
# -------------------------------------------
N_THREADS = max(1, os.cpu_count() or 2)
print("threads:", N_THREADS, flush=True)
def run(cmd, log=None):
    print("$ " + cmd, flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=W, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    p.wait()
    print(f"[exit {p.returncode} in {time.time()-t0:.0f}s]", flush=True)
    out = "".join(lines)
    if log:
        open(log, "w").write(out)
    if p.returncode != 0:
        raise RuntimeError("FAILED: " + cmd)
    return out
run("python3 --version && (nvidia-smi -L || echo no-gpu) && df -h /kaggle/working | tail -1")
run("pip install -q --no-cache-dir -U huggingface_hub datasets gguf")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or None
except Exception as e:
    print("secrets unavailable:", e, "(Add-ons -> Secrets -> HF_TOKEN -> attach)")
    HF_TOKEN = None
print("HF_TOKEN:", "present" if HF_TOKEN else "MISSING (public snapshot still works; HF upload will fail)")


In [ ]:
# [1] Base model snapshot
from huggingface_hub import snapshot_download
if not (os.path.isdir("base") and os.path.exists("base/config.json")):
    snapshot_download(repo_id=BASE_REPO, local_dir="base", token=HF_TOKEN, allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.model", "*.txt", "*.md"])
arch = json.load(open("base/config.json")).get("architectures", [])
print("architectures:", arch)
assert any("Qwen3" in a for a in arch), "WRONG BASE MODEL for this pipeline: " + str(arch)
n = sum(1 for _, _, fs in os.walk("base") for f in fs)
print("base files:", n)


In [ ]:
# [2] Build llama.cpp (skip if already built)
BIN = "llama.cpp/build/bin"
NEED = ["llama-quantize", "llama-imatrix", "llama-perplexity"]
if all(os.path.exists(BIN + "/" + b) for b in NEED):
    print("llama.cpp already built - skipping")
else:
    if not os.path.isdir("llama.cpp"):
        run("git clone --depth 1 https://github.com/ggml-org/llama.cpp")
    run("cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release")
    run("cmake --build llama.cpp/build -j " + str(N_THREADS))
    assert all(os.path.exists(BIN + "/" + b) for b in NEED), "build incomplete"
print("build OK")


In [ ]:
# [3] Convert HF -> GGUF F16, then delete the 5.9GB base shards (config + tokenizer kept)
if not (os.path.exists("smol-f16.gguf") and os.path.getsize("smol-f16.gguf") > 5000000000):
    run("test -f llama.cpp/convert_hf_to_gguf.py && CV=llama.cpp/convert_hf_to_gguf.py || CV=llama.cpp/tools/convert_hf_to_gguf.py; python3 $CV base --outfile smol-f16.gguf --outtype f16")
    assert os.path.getsize("smol-f16.gguf") > 5000000000, "F16 convert looks incomplete"
    import glob as _g
    gone = 0
    for f in _g.glob("base/*.safetensors") + _g.glob("base/*.bin"):
        os.remove(f)
        gone += 1
    print(f"base shards deleted: {gone} files")
print("F16 OK:", round(os.path.getsize("smol-f16.gguf") / 1e9, 2), "GB")


In [ ]:
# [4] Calibration text: wikitext-2, deterministic seed-42 sample of 4000 lines
def load_wikitext2():
    from datasets import load_dataset
    try:
        return load_dataset("wikitext", "wikitext-2-raw-v1")
    except Exception as e:
        print("canonical wikitext failed, trying mirror:", e)
        return load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

if not (os.path.exists("calib.txt") and os.path.exists("wikitext-test.txt")):
    ds = load_wikitext2()
    train_lines = [t for t in ds["train"]["text"] if t and t.strip()]
    calib = random.Random(SEED).sample(train_lines, min(CALIB_LINES, len(train_lines)))
    open("calib.txt", "w").write("\n".join(calib) + "\n")
    test_lines = [t for t in ds["test"]["text"] if t and t.strip()][:128]
    open("wikitext-test.txt", "w").write("\n".join(test_lines) + "\n")  # 128 lines ~= 7min per ppl run; full 2891 lines = 141 chunks ~= 5h+ per model and guarantees timeout
    print("calib lines:", len(calib), "| test lines:", len(test_lines))
else:
    print("calib files exist - skipping")


In [ ]:
# [5] Importance matrix (LONG POLE, ~3h). Periodic saves to imatrix.dat.
BIN = "llama.cpp/build/bin"
if not (os.path.exists("imatrix.dat") and os.path.getsize("imatrix.dat") > 1000000):
    run(BIN + "/llama-imatrix -m smol-f16.gguf -f calib.txt -o imatrix.dat -c 512 -b 2048 -t " + str(N_THREADS) + " --output-frequency 8")
    assert os.path.getsize("imatrix.dat") > 1000000, "imatrix looks incomplete"
print("imatrix OK:", round(os.path.getsize("imatrix.dat") / 1e6, 1), "MB")


In [ ]:
# [6] Quantize with imatrix
BIN = "llama.cpp/build/bin"
for q in QUANTS:
    out = "smol-" + q + ".gguf"
    if not (os.path.exists(out) and os.path.getsize(out) > 1000000000):
        run(BIN + "/llama-quantize --imatrix imatrix.dat smol-f16.gguf " + out + " " + q)
    print(out, round(os.path.getsize(out) / 1e9, 2), "GB")


In [ ]:
# [7] Perplexity eval on wikitext-test (F16 vs quants)
BIN = "llama.cpp/build/bin"
JOBS = [("f16", "smol-f16.gguf", "ppl-f16.log"), ("Q4_K_M", "smol-Q4_K_M.gguf", "ppl-q4.log"), ("Q8_0", "smol-Q8_0.gguf", "ppl-q8.log")]
PPLS = {}
for tag, gguf, logf in JOBS:
    out = run(BIN + "/llama-perplexity -m " + gguf + " -f wikitext-test.txt -c 2048 -b 2048 -t " + str(N_THREADS), log=logf)
    m = re.findall(r"Final estimate:\s*PPL\s*=\s*([0-9.]+)", out)
    assert m, "no PPL found in " + logf
    PPLS[tag] = float(m[-1])
    print(tag, "PPL =", PPLS[tag])
rise = (PPLS["Q4_K_M"] - PPLS["f16"]) / PPLS["f16"] * 100
print(f"Q4 rise vs F16: {rise:+.2f}% (bar <{BAR_PCT}%)")


In [ ]:
# [8] Free output space (build tree + base leftovers), then verify + pack
shutil.rmtree("llama.cpp", ignore_errors=True)
shutil.rmtree("base", ignore_errors=True)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

ARTIFACTS = ["smol-Q4_K_M.gguf", "smol-Q8_0.gguf", "imatrix.dat", "calib.txt", "wikitext-test.txt"]
for f in ARTIFACTS:
    assert os.path.exists(f) and os.path.getsize(f) > 0, "missing " + f
SHAS = {f: sha256_file(f) for f in ARTIFACTS}
open("SHA256.txt", "w").write("".join(SHAS[f] + "  " + f + "\n" for f in ARTIFACTS))
repro = {
    "base_repo": BASE_REPO, "seed": SEED, "calib_lines": CALIB_LINES,
    "threads": N_THREADS, "quants": QUANTS, "ppl": PPLS,
    "rise_q4_pct": round((PPLS["Q4_K_M"] - PPLS["f16"]) / PPLS["f16"] * 100, 2),
    "sha16": {f: SHAS[f][:16] for f in ARTIFACTS},
    "sizes_gb": {f: round(os.path.getsize(f) / 1e9, 2) for f in ARTIFACTS},
}
json.dump(repro, open("REPRO.json", "w"), indent=2)
UPLOAD_FILES = ["smol-Q4_K_M.gguf", "smol-Q8_0.gguf", "imatrix.dat", "calib.txt", "REPRO.json", "SHA256.txt", "UPLOAD.txt"]
open("UPLOAD.txt", "w").write("repo: <username>/" + REPO_NAME + "\n" + "\n".join(UPLOAD_FILES) + "\n")
run("pip freeze | grep -Ei 'torch|transformers|datasets|safetensors|gguf|numpy|huggingface' | tee VERSIONS.txt")
run("ls -lh *.gguf *.dat *.txt *.json")
open("OUTPUT_MANIFEST.txt", "w").write("\n".join(f"{f}: {os.path.getsize(f)} bytes" for f in ARTIFACTS + ["smol-f16.gguf", "REPRO.json", "SHA256.txt"]))
print(open("OUTPUT_MANIFEST.txt").read())
print("PACK DONE")


In [ ]:
# [9] Gate on eval, then auto-upload to HF (non-interactive; gate failure stops loudly, uploads nothing)
import re
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

def read_ppl(path):
    m = re.findall(r"Final estimate:\s*PPL\s*=\s*([0-9.]+)", open(path).read())
    return float(m[-1]) if m else None

ppl_f16, ppl_q4 = read_ppl("ppl-f16.log"), read_ppl("ppl-q4.log")
assert ppl_f16 and ppl_q4, "eval logs missing - refusing to upload"
UPLOAD_FILES = ["smol-Q4_K_M.gguf", "smol-Q8_0.gguf", "imatrix.dat", "calib.txt", "REPRO.json", "SHA256.txt", "UPLOAD.txt"]
assert all(os.path.exists(f) and os.path.getsize(f) > 0 for f in UPLOAD_FILES), "missing upload files"
rise = (ppl_q4 - ppl_f16) / ppl_f16 * 100
print(f"F16={ppl_f16:.4f} Q4={ppl_q4:.4f} rise={rise:+.2f}% (bar <{BAR_PCT}%)")
if rise >= BAR_PCT:
    raise SystemExit("GATE FAILED - not uploading")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit("HF_TOKEN secret not attached (Add-ons -> Secrets -> HF_TOKEN -> attach)") from e

from huggingface_hub import HfApi
api = HfApi(token=token)
repo_id = api.whoami()["name"] + "/" + REPO_NAME
api.create_repo(repo_id, repo_type="model", exist_ok=True, private=REPO_PRIVATE)
for f in UPLOAD_FILES:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=repo_id,
                    repo_type="model", commit_message="add " + f + f" (Q4 rise {rise:+.2f}%)")
print("GATE PASSED - UPLOADED -> https://huggingface.co/" + repo_id)
